# Carrier Allocation Data Cleanup (Bronze -> Silver)

Reads the raw consolidated bronze delta table produced by
`consolidate_carrier_alloc.ipynb` and writes a cleaner silver delta table
for downstream consumption (Power BI, analyst queries, other notebooks).

**Responsibilities:**
- **Drop empty columns** -- columns where every row is null, empty, or
  whitespace. Detected dynamically from the data, not hard-coded, so the
  silver layer self-adjusts if a bronze column becomes populated later.
- **Drop business-rejected columns** -- an explicit hard-coded list for
  columns the business has marked as unneeded (`DROP_COLUMNS`).
- **Apply business-driven renames** -- semantic renames where the source
  headers are ambiguous or unnamed (`RENAME_MAP`), e.g. the unnamed 27th
  column is `count`.

**Why is this separate from the bronze notebook?**
Microsoft Fabric's medallion architecture recommends separating raw
ingestion (bronze) from business curation (silver). Keeping them separate
means the raw layer stays replayable from the source Excel files and is
never mutated by business-rule changes; only the silver layer needs to be
re-run when those rules change.


In [ ]:
# ============================================================
# CONFIGURATION - adjust these values for your environment
# ============================================================

# Input bronze table (produced by consolidate_carrier_alloc.ipynb).
BRONZE_TABLE_NAME = "consolidated_carrier_alloc"

# Output silver table written by this notebook.
SILVER_TABLE_NAME = "silver.consolidated_carrier_alloc_clean"

# Metadata columns added by the bronze ingestion notebook.
# These are always preserved through the silver transformation,
# even if a particular metadata column happens to be empty.
METADATA_COLUMNS = {
    "_source_file",
    "_sheet_name",
    "_carrier_code",
    "_alloc_date",
    "_ingested_at",
}

# Columns the business has explicitly marked as unneeded downstream.
# Names must match the snake_case form written to the bronze table
# (apply the same lowercasing rule as bronze's sanitize_columns_for_delta).
DROP_COLUMNS = {
    "note_for_sandeep",
    "addl_internal_note",
}

# Business-driven renames: bronze column name -> silver column name.
# Applied after empty/business drops, so keys should only reference
# columns that survive those drops.
RENAME_MAP = {
    "_col_27": "count",
}


In [ ]:
import logging

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("carrier_alloc_clean")


In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================


def find_empty_columns(df: DataFrame, preserve=None):
    """Return the names of columns where every row is null, empty, or whitespace.

    Runs a single aggregation pass -- one Spark job regardless of how many
    columns the DataFrame has -- by building a conditional sum expression per
    candidate column and collecting all counts in one round-trip.

    Casting each column to string before the trim/compare means the check is
    uniform across numeric, date, and string columns: a numeric 0 still counts
    as non-empty (it stringifies to "0"), and nulls are excluded explicitly.

    Args:
        df:       the Spark DataFrame to analyze.
        preserve: optional iterable of column names to exclude from the scan
                  (e.g., metadata columns that must be kept even when empty).

    Returns:
        A list of column names with zero non-empty values.
    """
    preserve = set(preserve or ())
    candidates = [c for c in df.columns if c not in preserve]
    if not candidates:
        return []

    non_empty_counts = [
        F.sum(
            F.when(
                F.col(c).isNotNull()
                & (F.trim(F.col(c).cast("string")) != ""),
                1,
            ).otherwise(0)
        ).alias(c)
        for c in candidates
    ]
    counts = df.agg(*non_empty_counts).collect()[0].asDict()
    return [c for c, n in counts.items() if n == 0]


def drop_columns_safely(df: DataFrame, columns_to_drop):
    """Drop the specified columns, logging any that are not in the schema.

    Spark's DataFrame.drop() is already tolerant of missing columns, but doing
    the membership check ourselves lets us warn when the silver config drifts
    out of sync with the bronze schema -- e.g., a DROP_COLUMNS entry was
    removed from bronze or mistyped.
    """
    columns_to_drop = set(columns_to_drop)
    present = [c for c in df.columns if c in columns_to_drop]
    missing = columns_to_drop - set(present)
    if missing:
        log.warning(
            "Configured drop targets not present in bronze schema: %s",
            sorted(missing),
        )
    return df.drop(*present) if present else df


def apply_renames(df: DataFrame, rename_map):
    """Rename columns according to rename_map using a single select() pass.

    One select() call is cheaper than chained withColumnRenamed() calls
    (single analyzed plan, no intermediate projections) and makes the
    transformation trivial to reason about: every column is rewritten
    exactly once, in order, via its mapped alias or its original name.

    Unknown keys in rename_map are logged as warnings rather than raising,
    so the silver config can tolerate bronze schema changes gracefully.
    """
    missing = [k for k in rename_map if k not in df.columns]
    if missing:
        log.warning(
            "Configured rename keys not present in bronze schema: %s", missing
        )
    return df.select(
        *[F.col(c).alias(rename_map.get(c, c)) for c in df.columns]
    )


In [ ]:
# ============================================================
# MAIN PROCESSING
# ============================================================

# ----- Read bronze -----
bronze_df = spark.read.table(BRONZE_TABLE_NAME)
log.info(
    "Read %d rows x %d columns from bronze table '%s'",
    bronze_df.count(),
    len(bronze_df.columns),
    BRONZE_TABLE_NAME,
)

# ----- Detect empty columns dynamically (preserving metadata) -----
empty_cols = find_empty_columns(bronze_df, preserve=METADATA_COLUMNS)
log.info("Empty columns detected: %d", len(empty_cols))
for c in sorted(empty_cols):
    log.info("  - %s", c)

# ----- Drop empty + business-rejected columns in one pass -----
drop_targets = set(empty_cols) | DROP_COLUMNS
silver_df = drop_columns_safely(bronze_df, drop_targets)
log.info(
    "Dropped %d columns (%d empty + %d business-rejected)",
    len(bronze_df.columns) - len(silver_df.columns),
    len(empty_cols),
    len(DROP_COLUMNS & set(bronze_df.columns)),
)

# ----- Apply business renames -----
silver_df = apply_renames(silver_df, RENAME_MAP)

# ----- Add silver ingestion timestamp and write -----
silver_df = silver_df.withColumn("_silver_ingested_at", F.current_timestamp())

(
    silver_df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE_NAME)
)

log.info(
    "Saved %d rows x %d columns to silver delta table '%s'",
    silver_df.count(),
    len(silver_df.columns),
    SILVER_TABLE_NAME,
)


In [ ]:
# ============================================================
# VERIFY RESULTS
# ============================================================

df = spark.read.table(SILVER_TABLE_NAME)
print(f"Table:   {SILVER_TABLE_NAME}")
print(f"Rows:    {df.count()}")
print(f"Columns: {len(df.columns)}")
print()
display(df.limit(20))
